In [14]:
!curl -L https://github.com/LucasDuarte026/electric_grid_graph/raw/main/data.zip -o data.zip
!unzip -o data.zip "data/*"
!rm data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  197k  100  197k    0     0  1471k      0 --:--:-- --:--:-- --:--:-- 1471k
Archive:  data.zip
   creating: data/
   creating: data/ieee14/
  inflating: data/ieee14/IEEE 14 bus.epc  
  inflating: data/ieee14/IEEE 14 bus.pwb  
  inflating: data/ieee14/IEEE 14 bus.pwd  
  inflating: data/ieee14/ieee14.raw  
   creating: data/ieee300/
  inflating: data/ieee300/ieee300.raw  
  inflating: data/ieee300/ieee300bus.epc  
  inflating: data/ieee300/ieee300bus.pwb  
  inflating: data/ieee300/ieee300bus.pwd  


In [ ]:
%pip install networkx matplotlib ipywidgets

In [ ]:
import math
import heapq
import networkx as nx
from collections import deque
import matplotlib.pyplot as plt
import time
import ipywidgets as widgets
from IPython.display import display

# Otimização de Rotas de Transmissão e Análise de Conectividade em Sistemas de Potência IEEE 14 e 300

## Introdução

A infraestrutura elétrica enfrenta desafios decorrentes da alta complexidade das redes e da necessidade de respostas rápidas a falhas operacionais. A manutenção da estabilidade do sistema e a garantia do fornecimento contínuo de energia, mesmo após a perda de componentes críticos, exigem algoritmos eficientes de busca e otimização.

Neste trabalho, são aplicados algoritmos de busca clássicos e informados, amplamente utilizados em inteligência artificial, para a análise de redes de transmissão de energia. Utilizam-se os sistemas de teste IEEE 14-Bus, como modelo exploratório, e IEEE 300-Bus, como cenário de maior escala e complexidade.

A escolha desses sistemas é fundamentada em sua ampla adoção na literatura técnica. O modelo de 14 barras representa uma versão simplificada de um sistema real, sendo adequado para validações iniciais. Já o sistema de 300 barras, desenvolvido pela IEEE Test Systems Task Force, apresenta maior complexidade estrutural, permitindo avaliar a escalabilidade e robustez dos algoritmos. Embora simplificados, ambos preservam características elétricas e topológicas essenciais.

Em sistemas reais, falhas em linhas de transmissão podem ocorrer devido a eventos climáticos, sobrecargas ou falhas de equipamentos. Nessas situações, é necessário verificar rapidamente se a rede permanece conectada ou se ocorreu ilhamento, além de identificar rotas alternativas para o fluxo de potência. Esse problema pode ser modelado como uma busca em grafos, onde os nós representam barramentos e as arestas representam conexões elétricas.

A relevância dessa análise está diretamente ligada à segurança energética e ao critério de contingência $N-1$, que exige que o sistema suporte a perda de qualquer componente individual sem interrupção do fornecimento. Para isso, são simuladas análises típicas de centros de controle (SCADA/EMS), utilizando três algoritmos principais:

- **DFS (Busca em Profundidade):** empregado para verificar a conectividade do grafo e identificar rapidamente a formação de ilhas elétricas após falhas.

- **BFS (Busca em Largura):** utilizado para encontrar caminhos com o menor número de conexões (hops), independentemente do custo elétrico.

- **A\***: algoritmo de busca informada que utiliza uma heurística baseada na distância euclidiana entre barramentos (a partir de suas coordenadas), permitindo encontrar caminhos eficientes ao equilibrar custo acumulado e estimativa até o destino.

A metodologia segue uma progressão de complexidade. Inicialmente, os algoritmos são aplicados ao sistema IEEE 14-Bus, permitindo análise detalhada e validação dos resultados em escala reduzida. Em seguida, são aplicados ao sistema IEEE 300-Bus, que impõe desafios adicionais, como o aumento do espaço de estados, maior densidade de conexões e maior dependência da qualidade da heurística.

As instruções para execução do código e dependências estão disponíveis no repositório do projeto.

---

## Modelagem do Sistema como Problema de Busca

A rede elétrica é convertida em um modelo computacional a partir do processamento de arquivos estruturados no formato `.raw`, utilizando a biblioteca NetworkX. O sistema é representado como um grafo, no qual os elementos físicos são mapeados da seguinte forma:

- **Espaço de Estados (Nós):** conjunto de barramentos do sistema. Cada nó é classificado como barramento de carga (*load*) ou de geração (*generator*), com base em atributos associados.

- **Estado Inicial e Objetivo:** o estado inicial corresponde, em geral, a um barramento de geração (como o *slack bus*), enquanto o objetivo é alcançar barramentos de carga considerados críticos.

- **Função de Transição (Arestas):** definida pelas conexões físicas entre barramentos, incluindo:
  - **Linhas de Transmissão (*lines*):** conexões diretas entre nós.
  - **Transformadores (*transformers*):** conexões que interligam diferentes níveis de tensão, processadas como arestas no grafo.

- **Custo da Transição (Peso):** definido pela reatância elétrica absoluta ($|x|$), atribuída como peso da aresta. Esse valor é utilizado por sua relação direta com a impedância da linha e sua influência no fluxo de potência e na estabilidade do sistema.

Essa modelagem permite que os algoritmos de busca operem sobre uma estrutura que preserva propriedades elétricas e topológicas relevantes, garantindo coerência entre o modelo computacional e o sistema físico.

---

## Implementação do Parser do Sistema

A conversão do arquivo `.raw` em um grafo utilizável pelos algoritmos de busca é realizada por meio de um parser desenvolvido em Python, utilizando a biblioteca NetworkX. Esse parser é responsável por interpretar a estrutura do arquivo e mapear corretamente os componentes elétricos para nós e arestas do grafo.

A função `parse_raw_system` realiza esse processamento de forma estruturada, percorrendo o arquivo linha a linha e identificando as diferentes seções de dados, como barramentos, linhas, transformadores, geradores e cargas.

### Estrutura Geral

O parser segue três etapas principais:

1. **Leitura e pré-processamento do arquivo:**
   - O arquivo é lido linha a linha.
   - Espaços em branco e cabeçalhos são ignorados até o início efetivo dos dados.

2. **Identificação de seções:**
   - O parser detecta dinamicamente as seções do arquivo com base em marcadores como:
     - `BEGIN BRANCH DATA`
     - `BEGIN TRANSFORMER DATA`
     - `BEGIN GENERATOR DATA`
     - `BEGIN LOAD DATA`
   - Isso permite adaptar o processamento conforme o tipo de dado sendo analisado.

3. **Construção do grafo:**
   - Um grafo não direcionado (`nx.Graph`) é utilizado para representar o sistema elétrico.
   - Nós e arestas são adicionados conforme os dados são interpretados.

### Mapeamento dos Componentes

- **Barramentos (Nós):**
  - Cada barramento é adicionado como um nó no grafo.
  - Todos os nós recebem inicialmente o atributo `type="bus"`.

- **Linhas de Transmissão (Arestas):**
  - Conexões entre barramentos são extraídas da seção de *branch*.
  - A reatância (`x`) é utilizada como peso da aresta:
    - `weight = |x|`
  - Essas arestas recebem o atributo `type="line"`.

- **Transformadores:**
  - Modelados como arestas, similar às linhas.
  - Como os dados de transformadores são distribuídos em múltiplas linhas no arquivo `.raw`, é necessário acumular essas informações em um buffer antes do processamento.
  - Após a leitura completa do bloco, a reatância é extraída e a aresta é adicionada com `type="transformer"`.

- **Geradores e Cargas:**
  - Os nós já existentes são enriquecidos com atributos adicionais:
    - `generator=True` para barramentos de geração
    - `load=True` para barramentos de carga

### Tratamento de Erros e Robustez

O parser utiliza blocos `try/except` para evitar falhas causadas por inconsistências ou variações no formato do arquivo. Linhas inválidas ou incompletas são ignoradas, garantindo maior robustez no processamento de diferentes versões de arquivos `.raw`.

### Exemplo de Uso

```G = parse_raw_system("./data/ieee14/ieee14.raw")```

---

A seguir, apresenta-se a implementação do parser descrito anteriormente:

In [ ]:
def parse_raw_system(filepath):
    """
    Robust parser for IEEE .raw files.
    Returns a NetworkX graph with rich attributes.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f]

    G = nx.Graph()
    section = "bus"  # Default initial section
    buffer = []
    started = False 

    def flush_transformer_block(block):
        """Processes multi-line transformer data to extract reactance."""
        try:
            # Extracting from/to nodes and the reactance (x) from the block
            u, v = int(block[0][0]), int(block[0][1])
            x = float(block[1][1])

            G.add_edge(u, v, weight=abs(x), type="transformer")
        except (IndexError, ValueError):
            pass

    for line in lines:
        # Locate the start of the data, skipping the file header
        if not started:
            if line and line[0].isdigit() and "'" in line:
                started = True
            else:
                continue

        # Detect section transitions within the .raw file
        if "BEGIN BRANCH DATA" in line:
            section = "branch"
            continue
        elif "BEGIN TRANSFORMER DATA" in line:
            section = "transformer"
            buffer = []
            continue
        elif "BEGIN GENERATOR DATA" in line:
            section = "generator"
            continue
        elif "BEGIN LOAD DATA" in line:
            section = "load"
            continue
        elif "END OF" in line:
            if section == "transformer" and buffer:
                flush_transformer_block(buffer)
            section = None
            continue

        # Skip empty lines or footer marks
        if not line or line.startswith("0 /"):
            continue

        parts = [p.strip() for p in line.split(',')]

        # Map buses as graph nodes
        if section == "bus":
            try:
                G.add_node(int(parts[0]), type="bus")
            except (IndexError, ValueError):
                continue

        # Map branches as edges using reactance as edge weight
        elif section == "branch":
            try:
                u, v, x = int(parts[0]), int(parts[1]), float(parts[4])
                G.add_edge(u, v, weight=abs(x), type="line")
            except (IndexError, ValueError):
                continue

        # Accumulate multi-line transformer data for batch processing
        elif section == "transformer":
            buffer.append(parts)
            if len(buffer) == 4:
                flush_transformer_block(buffer)
                buffer = []

        # Tag existing nodes as generators
        elif section == "generator":
            try:
                bus_id = int(parts[0])
                if bus_id in G.nodes:
                    G.nodes[bus_id]["generator"] = True
            except (IndexError, ValueError):
                continue

        # Tag existing nodes as loads
        elif section == "load":
            try:
                bus_id = int(parts[0])
                if bus_id in G.nodes:
                    G.nodes[bus_id]["load"] = True
            except (IndexError, ValueError):
                continue

    return G

# Load the system model
# G = parse_raw_system("./data/ieee14/ieee14.raw")
G = parse_raw_system("./data/IEEE 14-Bus System/IEEE 14 bus.raw")
# 

## Análise do Sistema IEEE 14-Bus

### Visualização e Validação do Grafo

Para validar a integridade da topologia importada, são utilizadas técnicas de visualização que permitem comparar o modelo computacional com os diagramas unifilares de referência do sistema IEEE 14-Bus. A visualização é realizada em duas etapas principais:

- **Layout Topológico:** utiliza-se o algoritmo `spring_layout` da biblioteca NetworkX, baseado no modelo de forças de Fruchterman-Reingold. Esse método posiciona os nós de forma a evidenciar a estrutura da rede, facilitando a identificação de agrupamentos e padrões de conectividade.

- **Renderização Customizada:** por meio da função `plot_power_network`, o grafo é exibido com informações adicionais, incluindo os valores de reatância ($|x|$) associados às arestas. A função também permite destacar caminhos específicos (por exemplo, em vermelho), o que é útil para validar visualmente os resultados dos algoritmos de busca e verificar a continuidade do fornecimento em cenários de falha.

A seguir, apresenta-se a implementação e aplicação das ferramentas de visualização para o sistema IEEE 14-Bus.

In [ ]:
def plot_small_network(graph, pos, title="Small Power Network"):
    """
    Optimized visualization for small-scale networks. Focuses on readability, node identification, and connectivity.
    """
    plt.figure(figsize=(12, 8))
    
    # Edges are slightly thicker
    nx.draw_networkx_edges(graph, pos, width=1.5, alpha=0.5, edge_color="#34495e")
    
    # Node color based on degree
    node_color = [graph.degree(n) for n in graph.nodes]
    
    # Larger nodes for better label background
    nodes = nx.draw_networkx_nodes(
        graph, pos, 
        node_size=700, 
        node_color=node_color, 
        cmap=plt.cm.viridis, 
        linewidths=2,
        edgecolors='white' # Adds a border to nodes for better contrast
    )
    
    # For small networks, labels are important
    nx.draw_networkx_labels(graph, pos, font_size=10, font_weight="bold", font_color="black")
    
    plt.colorbar(nodes, label='Node Degree (Connections)')
    plt.title(title, fontsize=14, pad=20)
    plt.axis("off")
    plt.show()
    
pos = nx.spring_layout(G, seed=138) # seed=138 is choosen so the visualization is the same for every run
plot_small_network(G, pos, title="IEEE 14-Bus - Topology Validation")

Enquanto a função anterior é voltada à análise estrutural da rede, a função a seguir introduz um recurso essencial para a validação dos algoritmos de busca: o destaque de caminhos no grafo.

A função `plot_small_power_network` permite visualizar não apenas a topologia do sistema, mas também rotas específicas encontradas pelos algoritmos (DFS, BFS e A*). Para isso, um caminho pode ser fornecido como entrada e será destacado visualmente no grafo.

Além disso, a função inclui a exibição dos valores de reatância ($|x|$) nas arestas, permitindo uma análise mais detalhada das rotas em termos elétricos.

Essa abordagem é particularmente útil para:
- Validar se o caminho encontrado é contínuo e viável
- Comparar diferentes estratégias de busca
- Analisar qualitativamente o impacto do custo elétrico nas rotas escolhidas

In [ ]:
def plot_small_power_network(graph, pos, path=None, title="Network Overview"):
    """
    Visualizes the electrical graph. If a path is provided, highlights it in red to validate search results.
    """
    plt.figure(figsize=(10, 8))
    
    # Draw base nodes, edges and identification labels
    nx.draw_networkx_nodes(graph, pos, node_color="lightblue", node_size=500)
    nx.draw_networkx_labels(graph, pos, font_size=9, font_weight="bold")
    nx.draw_networkx_edges(graph, pos, edge_color="black", alpha=0.7)

    # Extract and format reactance values (|x|) as edge labels
    edge_labels = nx.get_edge_attributes(graph, 'weight')
    edge_labels_fmt = {k: f"{v:.4f}" for k, v in edge_labels.items()}
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels_fmt, font_size=7)

    # If a solution path is found, highlight it in red
    if path and all(node in graph for node in path):
        path_edges = list(zip(path, path[1:]))
        nx.draw_networkx_nodes(
            graph, pos, nodelist=path,
            node_color="#e74c3c", edgecolors='white',
            node_size=550, linewidths=1.5
        )
        nx.draw_networkx_edges(
            graph, pos, edgelist=path_edges,
            edge_color="#c0392b", width=3.0, alpha=0.9
        )

    plt.title(title)
    plt.axis("off")
    plt.show()

# Validating the topology with a sample path
plot_small_power_network(G, pos, path=[1, 2, 4, 7, 8], title="IEEE 14-Bus - Topology and Path Validation")

A visualização gerada confirma que a modelagem foi bem-sucedida com base em três aspectos principais:

- **Integridade Topológica:** a disposição dos nós e o número de conexões (arestas) estão consistentes com as especificações do sistema IEEE 14-Bus.

- **Consistência dos Dados:** os pesos das arestas refletem corretamente os valores de reatância ($|x|$) extraídos do arquivo `.raw`, garantindo coerência física na avaliação dos custos das rotas.

- **Validação de Caminhos:** o destaque visual de trajetórias permite verificar se as rotas obtidas pelos algoritmos são contínuas e válidas, assegurando a ausência de desconexões ou inconsistências estruturais.

Com a topologia devidamente validada, o sistema torna-se apto para a aplicação dos algoritmos de busca **DFS**, **BFS** e **A\***, no contexto da análise de contingência $N-1$.

### Buscas Não-Informadas

As buscas não informadas são utilizadas para analisar propriedades estruturais da rede, como conectividade, sem considerar custos elétricos. No contexto de sistemas de potência, essas abordagens são úteis para verificar se o sistema permanece conectado após uma contingência, evitando o isolamento de cargas críticas.

#### Busca em Profundidade (DFS)

O algoritmo de busca em profundidade (DFS) é aplicado para verificar a conectividade da rede e identificar a possível formação de ilhas elétricas. Em sistemas de potência, a abertura de elementos como disjuntores pode desconectar partes da rede, comprometendo o fornecimento de energia.

- **Raciocínio:** o algoritmo explora recursivamente os nós adjacentes, seguindo um caminho até seu limite antes de retroceder (*backtracking*).

- **Aplicação:** a partir de um barramento de geração, o DFS verifica se determinados barramentos de carga são alcançáveis. Caso não sejam, caracteriza-se a formação de ilhas na rede.

- **Vantagem:** apresenta baixa complexidade computacional ($O(V + E)$), sendo eficiente para análise de conectividade mesmo em redes de grande porte.

A implementação do algoritmo **DFS** utiliza uma pilha (*stack*) para controlar a exploração dos nós. Cada elemento da pilha representa um caminho parcial, permitindo a reconstrução da rota desde o nó inicial até o objetivo.

O algoritmo explora profundamente cada ramo antes de retroceder, retornando o primeiro caminho encontrado entre o barramento de origem e o destino. Como não considera custos, o caminho retornado não é necessariamente o mais curto ou ótimo.

A seguir, apresenta-se a implementação da estratégia.

In [ ]:
def dfs_search(graph, start, goal):
    """
    Depth-first search implementation.
    """
    # Check if start and goal are in graph
    if start not in graph or goal not in graph:
        return []
    
    if start == goal: 
        return [start]
    
    # Stack stores paths to allow backtracking and path reconstruction
    visited = set()
    stack = [[start]]
    
    while stack:
        # Pop the last path added (LIFO behavior)
        path = stack.pop()
        node = path[-1]
        
        if node not in visited:
            visited.add(node)
            
            if node == goal: 
                return path
            
            # Explore neighbors; reversed to maintain order if needed
            for neighbor in reversed(list(graph.neighbors(node))):
                if neighbor not in visited:
                    # Append new path to the stack
                    stack.append(path + [neighbor])
    return []

Como exemplo de aplicação, considera-se a busca de um caminho entre o nó 1 e o nó 8. O resultado obtido pelo algoritmo é então validado visualmente por meio da função `plot_small_power_network`, permitindo verificar a continuidade e coerência da rota encontrada.

In [ ]:
path=dfs_search(G, 1, 14)
plot_small_power_network(G, pos, path, title="IEEE 14-Bus - DFS-Path from 1 to 14")

#### Busca em Largura (BFS)

Diferente do DFS, o algoritmo de busca em largura (BFS) prioriza a distância topológica entre os nós. Ele explora a rede em níveis, garantindo a identificação do caminho com o menor número de conexões (saltos) entre a origem e o destino.

- **Raciocínio:** o algoritmo visita todos os nós vizinhos diretos antes de avançar para o próximo nível de profundidade, caracterizando uma exploração em camadas.

- **Aplicação:** é utilizado para determinar rotas mínimas em termos de número de arestas, sendo útil para análises estruturais da rede, independentemente dos parâmetros elétricos.

- **Vantagem:** garante a obtenção do caminho com o menor número de conexões, com complexidade computacional $O(V + E)$.

A implementação do BFS utiliza uma fila (*queue*) baseada em `deque`, garantindo a exploração em ordem de níveis. Cada elemento da fila representa um caminho parcial, permitindo reconstruir a rota entre o nó inicial e o objetivo.

Diferentemente do DFS, o BFS assegura que o primeiro caminho encontrado possui o menor número de conexões, sendo adequado para análises topológicas da rede.

A seguir, apresenta-se a implementação da estratégia.

In [ ]:
def bfs_search(graph, start, goal):
    """
    Breadth-first search using a double-ended queue (deque).
    """
    # Check if start and goal are in graph
    if start not in graph or goal not in graph:
        return []
    
    # Tracking visited nodes at discovery to prevent redundant cycles
    visited = {start}
    # Queue stores paths; popleft() ensures level-order traversal (FIFO)
    queue = deque([[start]])
    
    if start == goal: return [start]
    
    while queue:
        # Get the oldest path from the queue
        path = queue.popleft()
        node = path[-1]
        
        # Explore all immediate neighbors before moving deeper
        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                visited.add(neighbor)
                new_path = path + [neighbor]
                
                # Goal check at discovery for better performance
                if neighbor == goal: 
                    return new_path
                
                queue.append(new_path)
    return []

Como exemplo de aplicação, considera-se, novamente, a busca de um caminho entre o nó 1 e o nó 8. O resultado obtido pelo algoritmo é então validado visualmente por meio da função `plot_small_power_network`, permitindo verificar a continuidade e coerência da rota encontrada.

In [ ]:
path=bfs_search(G, 1, 14)
plot_small_power_network(G, pos, path, title="IEEE 14-Bus - BFS-Path from 1 to 14")

### Comparação entre DFS e BFS

Os algoritmos DFS e BFS apresentam comportamentos distintos na exploração da rede, refletindo diferentes objetivos de análise.

O **DFS** é mais adequado para verificar conectividade e identificar a existência de caminhos entre barramentos, sendo útil na detecção de ilhamento. No entanto, por explorar profundamente cada ramo, não garante a obtenção do caminho mais curto.

Por outro lado, o **BFS** explora a rede em camadas, assegurando a identificação do caminho com o menor número de conexões. Essa característica o torna mais apropriado para análises estruturais em que a minimização do número de componentes envolvidos é relevante.

Ambos os algoritmos desconsideram os parâmetros elétricos da rede, como a reatância das linhas, o que limita sua aplicação em cenários onde o custo elétrico das rotas é determinante. Essa limitação motiva o uso de algoritmos de busca informada, como o **A$^\ast$**, apresentado na próxima seção.

---

### Busca Informada

Diferentemente das buscas não informadas, os algoritmos de busca informada utilizam conhecimento adicional sobre o problema para guiar a exploração do espaço de estados. No contexto de sistemas de potência, isso permite considerar não apenas a conectividade da rede, mas também os custos associados às rotas, como a reatância das linhas.

Essa abordagem é particularmente relevante quando o objetivo não é apenas encontrar um caminho válido, mas sim identificar rotas mais eficientes do ponto de vista elétrico.

#### Busca A\*

O algoritmo A* combina o custo acumulado desde o nó inicial com uma estimativa do custo restante até o objetivo, sendo definido pela função:

$f(n) = g(n) + h(n)$

onde:
- $g(n)$ representa o custo real do caminho (soma das reatâncias)
- $h(n)$ é a heurística baseada na distância euclidiana entre os barramentos

Essa abordagem permite direcionar a busca para regiões mais promissoras do grafo, reduzindo a exploração de caminhos subótimos.

Diferentemente do BFS, que considera apenas o número de conexões, o A* incorpora parâmetros elétricos, resultando em rotas mais eficientes do ponto de vista físico.

In [ ]:
def astar_search(graph, start, goal, pos):
    """
    A* search using reactance as cost and Euclidean distance as heuristic.
    """
    # Check if start and goal are in graph
    if start not in graph or goal not in graph:
        return []
    
    # Priority queue: (f_score, node)
    open_list = [(0, start)]
    g_score = {node: float('inf') for node in graph}
    g_score[start] = 0
    parent = {start: None}
    closed_set = set()

    while open_list:
        # Pop node with the best f_score (lowest g + h)
        current_f, current_node = heapq.heappop(open_list)
        
        if current_node in closed_set:
            continue
        closed_set.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = parent[current_node]
            return path[::-1]

        for neighbor in graph.neighbors(current_node):
            # g(n): Actual reactance cost from start to neighbor
            w = graph[current_node][neighbor].get('weight', 0.1)
            tg = g_score[current_node] + w

            if tg < g_score[neighbor]:
                parent[neighbor] = current_node
                g_score[neighbor] = tg
                
                # h(n): Euclidean distance heuristic using node positions
                up, vp = pos[neighbor], pos[goal]
                h = math.sqrt((up[0]-vp[0])**2 + (up[1]-vp[1])**2) * 0.1
                
                # f(n) = g(n) + h(n)
                heapq.heappush(open_list, (tg + h, neighbor))
    return []

Como exemplo de aplicação, considera-se, novamente, a busca de um caminho entre o nó 1 e o nó 8. O resultado obtido pelo algoritmo é então validado visualmente por meio da função `plot_small_power_network`, permitindo verificar a continuidade e coerência da rota encontrada.

In [ ]:
path=astar_search(G, 1, 14, pos)
plot_small_power_network(G, pos, path, title="IEEE 14-Bus - A*-Path from 1 to 8")

### Comparação entre BFS e A*

A comparação entre os algoritmos BFS e A* evidencia a importância da consideração de custos elétricos na análise de rotas em sistemas de potência.

O **BFS** identifica caminhos com o menor número de conexões, sendo eficiente para análises puramente topológicas. No entanto, por não considerar os pesos das arestas, pode selecionar rotas com maior reatância total, o que implica maior impedância ao fluxo de potência.

Por outro lado, o **A*** incorpora o custo elétrico por meio da reatância das linhas ($|x|$), além de utilizar uma heurística baseada na distância euclidiana para guiar a busca. Como resultado, tende a encontrar caminhos com menor custo total, mesmo que envolvam um número maior de conexões.

Essa diferença é particularmente relevante em sistemas de potência, onde o desempenho da rede não depende apenas da conectividade, mas também das características elétricas dos componentes. Assim, enquanto o BFS é adequado para validações estruturais rápidas, o A* se mostra mais apropriado para análises operacionais que buscam eficiência no fluxo de potência.

Dessa forma, a utilização de buscas informadas representa um avanço significativo em relação às abordagens não informadas, permitindo uma modelagem mais fiel às condições físicas do sistema elétrico.

---

### Interface Interativa

Com o objetivo de facilitar a análise dos algoritmos e a realização de testes experimentais, foi desenvolvida uma interface interativa para execução das buscas no sistema elétrico modelado.

A interface permite ao usuário selecionar dinamicamente:
- o barramento de origem (*start bus*)
- o barramento de destino (*goal bus*)
- o algoritmo de busca a ser utilizado (DFS, BFS ou A*)

Além disso, são exibidas informações relevantes para a análise, incluindo:
- o caminho encontrado
- o tempo de execução
- o custo total da rota, calculado a partir da soma das reatâncias

A visualização gráfica associada permite validar qualitativamente os resultados, destacando o caminho obtido sobre a topologia da rede.

Essa abordagem interativa possibilita comparar diretamente o comportamento dos algoritmos em diferentes cenários, evidenciando as diferenças entre buscas não informadas e informadas em termos de eficiência e custo elétrico.

In [ ]:
class SearchInterface:
    """
    Interactive UI for power system pathfinding.
    Connects the GUI elements with the search algorithms and visualization.
    """
    def __init__(self, graph, pos, nodes):
        self.G = graph
        self.nodes = nodes
        self.pos = pos
        self.out = widgets.Output()
        
        # Dropdown menus for starting point, destination, and algorithm choice
        self.dropdown_start = widgets.Dropdown(options=nodes, description='Start Bus:', value=nodes[0])
        self.dropdown_goal = widgets.Dropdown(options=nodes, description='Goal Bus:', value=nodes[-1])
        self.dropdown_algo = widgets.Dropdown(
            options=['BFS (Uninformed)', 'DFS (Uninformed)', 'A* (Informed)'],
            description='Algorithm:',
            value='BFS (Uninformed)'
        )
        
        # Execution button with a visual success style
        self.btn_run = widgets.Button(description='Run Search', button_style='success')
        self.btn_run.on_click(self.run_search)
        
        # Layout organization using Horizontal and Vertical boxes
        self.ui = widgets.VBox([
            widgets.HBox([self.dropdown_start, self.dropdown_goal]), 
            self.dropdown_algo, 
            self.btn_run
        ])

    def run_search(self, b):
        """Callback function to execute the selected search algorithm."""
        with self.out:
            self.out.clear_output(wait=True)
            start_node = self.dropdown_start.value
            goal_node = self.dropdown_goal.value
            algo = self.dropdown_algo.value
            
            # Performance benchmarking: start timer
            t_start = time.time()

            path = []
            # Algorithm selection logic
            if algo == 'BFS (Uninformed)':
                path = bfs_search(self.G, start_node, goal_node)
            elif algo == 'DFS (Uninformed)':
                path = dfs_search(self.G, start_node, goal_node)
            elif algo == 'A* (Informed)':
                # For this, ensure pos is globally defined or passed here
                path = astar_search(self.G, start_node, goal_node, pos)

            t_end = time.time()
            execution_time = (t_end - t_start) * 1000

            if path:
                # Path formatting for display
                path_str = ' -> '.join(map(str, path))
                print(f"Path Found: {path_str}")
                print(f"Execution time: {execution_time:.4f} ms")
                
                # Calculate and display total reactance cost for any found path
                cost = sum(self.G[path[i]][path[i+1]]['weight'] for i in range(len(path)-1))
                print(f"Total Reactance: {cost:.4f} p.u.")
                
                # Trigger the visualization function
                plot_small_power_network(self.G, self.pos, path=path, title=f"Results: {algo}")
            else:
                print("[ERROR] No path found between the selected nodes.")

# Initialization and Display
search_app = SearchInterface(G, pos, list(G.nodes))
display(search_app.ui, search_app.out)

### Testes de Contingência e Validação (Cenário N-1)

Nesta seção, avalia-se a robustez do sistema por meio da simulação de falhas na rede de transmissão. O critério de contingência $N-1$ estabelece que o sistema deve continuar operando adequadamente mesmo após a perda de um único componente, como uma linha de transmissão ou transformador.

A metodologia adotada consiste na remoção de uma aresta do grafo, representando a indisponibilidade de uma linha, seguida da aplicação do algoritmo A* para determinar uma nova rota entre os barramentos de interesse. Essa abordagem permite verificar se o sistema permanece conectado e se existe uma alternativa viável para o fluxo de potência.

Além de identificar a nova rota, são avaliados:
- o custo elétrico da solução, com base na soma das reatâncias ($|x|$)
- o tempo de execução necessário para o redirecionamento
- a ocorrência de ilhamento, caso não exista caminho entre origem e destino

A seguir, apresenta-se a implementação do procedimento de teste de contingência.

In [ ]:
def run_contingency_test(graph, start, goal, edge_to_remove, pos):
    """
    Simulates a line failure (N-1 contingency) and attempts to reroute power using the A* algorithm.
    """
    # Create a copy to avoid modifying the original topology
    G_test = graph.copy()
    u_rem, v_rem = edge_to_remove

    # Check if the edge exists before attempting removal
    if G_test.has_edge(u_rem, v_rem):
        G_test.remove_edge(u_rem, v_rem)
        print(f"SIMULATED FAILURE: Line {u_rem}-{v_rem} is OUT OF SERVICE\n")
    else:
        print(f"[WARNING] Edge {u_rem}-{v_rem} not found in the current topology.")

    # Benchmark rerouting performance
    start_time = time.time()
    # Using the informed search to find the new optimal path
    path = astar_search(G_test, start, goal, pos) 
    end_time = time.time()

    if path:
        # Calculate new electrical cost (Total Impedance)
        cost = sum(G_test[path[i]][path[i+1]]['weight'] for i in range(len(path)-1))
        
        print(f"New Optimal Route Found (A*): {' -> '.join(map(str, path))}")
        print(f"New Total Reactance: {cost:.4f} p.u.")
        print(f"Rerouting execution: {(end_time - start_time)*1000:.4f} ms")
        
        # Visualize the network state after the fault and the new solution
        plot_small_power_network(G_test, pos, path=path, title=f"Contingency: Removal of Line {u_rem}-{v_rem}")
    else:
        # Critical alert if the contingency causes a blackout/island for the goal node
        print("[CRITICAL] Islanding detected: Goal is unreachable after this failure!")

# Practical example with a failure on a key line in the IEEE 14-Bus system
run_contingency_test(G, 1, 8, (7, 4), pos)

A simulação demonstra a capacidade do algoritmo A* de identificar rotas alternativas após a falha de um componente crítico da rede. Observa-se que, mesmo com a remoção da linha, o sistema pode manter a conectividade entre os barramentos, desde que existam caminhos redundantes.

Além disso, a nova rota pode apresentar um custo elétrico superior, refletindo o impacto da contingência na eficiência do sistema. Em casos onde nenhum caminho alternativo é encontrado, caracteriza-se a formação de ilhas elétricas, evidenciando uma violação do critério $N-1$.

## Análise do Sistema IEEE 300-Bus

Após a validação dos algoritmos no sistema IEEE 14-Bus, estende-se a análise para o sistema IEEE 300-Bus, que apresenta maior complexidade estrutural e dimensional.

Esse modelo permite avaliar o desempenho dos algoritmos em um cenário mais próximo de aplicações reais, caracterizado por um maior número de barramentos, maior densidade de conexões e um espaço de busca significativamente ampliado. Dessa forma, torna-se possível analisar aspectos como escalabilidade, eficiência computacional e impacto da heurística em redes de grande porte.

In [ ]:
# Load the system model
G = parse_raw_system("./data/IEEE 300-Bus System/ieee300bus.raw")

### Ajuste da Visualização para Redes de Grande Porte

Devido ao aumento significativo do número de nós e conexões no sistema IEEE 300-Bus, tornou-se necessário adaptar a estratégia de visualização. Em redes de alta densidade, a utilização dos mesmos parâmetros aplicados ao sistema IEEE 14-Bus compromete a legibilidade, gerando sobreposição excessiva de elementos.

Assim, foi desenvolvida uma função específica para redes de grande porte, com ajustes no tamanho dos nós, transparência das arestas e escala geral do layout. Essas modificações permitem preservar a percepção da estrutura global da rede, mantendo a visualização informativa mesmo em cenários de alta complexidade.

In [ ]:
def plot_large_network(graph, pos, title="Large Scale Network"):
    """
    Optimized visualization for high-density graphs.
    """
    plt.figure(figsize=(16, 10))
    
    # Draw edges with low alpha to emphasize paths over individual lines
    nx.draw_networkx_edges(graph, pos, width=0.8, alpha=0.2, edge_color="#2c3e50")
    
    # Color nodes based on connectivity (degree centrality)
    node_color = [graph.degree(n) for n in graph.nodes]
    
    nodes = nx.draw_networkx_nodes(
        graph, pos, 
        node_size=50, 
        node_color=node_color, 
        cmap=plt.cm.viridis, # Color scale from low to high connectivity
        alpha=0.9
    )
    
    plt.colorbar(nodes, label='Node Degree (Connections)')
    plt.title(title, fontsize=15)
    plt.axis("off")
    plt.show()

pos = nx.spring_layout(G, k=0.15, seed=138)
    
# Execution
plot_large_network(G, pos, "IEEE 300-Bus - Topology Validation")

### Destaque de Caminhos em Redes de Grande Escala

Para complementar a visualização em redes de grande porte, foi desenvolvida uma função específica para o destaque de caminhos encontrados pelos algoritmos de busca. Em sistemas densos como o IEEE 300-Bus, a exibição completa da topologia pode gerar poluição visual, dificultando a identificação de rotas relevantes.

A abordagem adotada utiliza uma representação suavizada da rede (*ghost layer*), com nós e arestas em tons claros e baixa opacidade, enquanto o caminho de interesse é destacado com cores mais intensas e maior espessura. Essa estratégia melhora significativamente o contraste visual, permitindo analisar de forma clara as soluções obtidas mesmo em grafos de alta complexidade.

In [ ]:
def plot_large_power_network(graph, pos, path=None, title="IEEE 300-Bus - Topology and Path Validation"):
    """
    Advanced visualization for large-scale power systems.
    Uses a 'ghost layer' for the base topology and highlights the search result in vibrant red for high-contrast analysis.
    """
    plt.figure(figsize=(16, 10))

    # Light gray edges and nodes to provide context without visual clutter
    nx.draw_networkx_nodes(graph, pos, node_color="#ecf0f1", node_size=40, 
                           edgecolors='#bdc3c7', linewidths=0.5, alpha=0.6)
    
    nx.draw_networkx_edges(graph, pos, edge_color="#bdc3c7", alpha=0.25, width=0.5)

    if path:
        path_edges = list(zip(path, path[1:]))
        
        # Highlight path nodes
        nx.draw_networkx_nodes(graph, pos, nodelist=path, node_color="#e74c3c", 
                               node_size=80, edgecolors='white', linewidths=1.5)
        
        # Highlight path edges
        nx.draw_networkx_edges(graph, pos, edgelist=path_edges, 
                               edge_color="#c0392b", width=3.0, alpha=0.9)
        
        # Selective labeling
        endpoints = {path[0]: f"{path[0]}", path[-1]: f"{path[-1]}"}
        nx.draw_networkx_labels(graph, pos, labels=endpoints, font_size=8, 
                                font_weight="bold", font_color="#2c3e50")

    plt.title(title, fontsize=16, pad=25)
    plt.axis("off")
    plt.show()
    
    
plot_large_power_network(G, pos, path=[1, 3, 150, 130, 129, 126, 125, 123, 124, 116, 120])

### Interface Interativa para Redes de Grande Porte

Para permitir a análise prática no sistema IEEE 300-Bus, foi desenvolvida uma versão adaptada da interface interativa, considerando as particularidades de redes de grande escala.

Assim como na versão para o sistema IEEE 14-Bus, a interface possibilita a seleção dos barramentos de origem e destino, bem como do algoritmo de busca (DFS, BFS ou A*). No entanto, devido à maior complexidade da rede, a visualização foi integrada a uma função específica que prioriza clareza e contraste na exibição dos resultados.

A execução de cada busca fornece:
- o caminho encontrado entre os barramentos
- o tempo de execução do algoritmo
- o custo total da rota, baseado na soma das reatâncias

Os resultados são apresentados graficamente com destaque para o caminho obtido, permitindo analisar o comportamento dos algoritmos em um cenário de maior escala. Essa abordagem possibilita avaliar, de forma interativa, aspectos como desempenho, escalabilidade e impacto da heurística na qualidade das soluções.

Dessa forma, a interface atua como uma ferramenta de apoio à validação experimental, evidenciando as diferenças entre os métodos de busca em um contexto mais próximo de aplicações reais.

In [ ]:
class LargeSearchInterface:
    """
    Interactive UI for power system pathfinding.
    Connects the GUI elements with the search algorithms and visualization.
    """
    def __init__(self, graph, pos, nodes):
        self.G = graph
        self.pos = pos
        self.nodes = nodes
        self.out = widgets.Output()
        
        # Dropdown menus for starting point, destination, and algorithm choice
        self.dropdown_start = widgets.Dropdown(options=nodes, description='Start Bus:', value=nodes[0])
        self.dropdown_goal = widgets.Dropdown(options=nodes, description='Goal Bus:', value=nodes[-1])
        self.dropdown_algo = widgets.Dropdown(
            options=['BFS (Uninformed)', 'DFS (Uninformed)', 'A* (Informed)'],
            description='Algorithm:',
            value='BFS (Uninformed)'
        )
        
        # Execution button with a visual success style
        self.btn_run = widgets.Button(description='Run Search', button_style='success')
        self.btn_run.on_click(self.run_search)
        
        # Layout organization using Horizontal and Vertical boxes
        self.ui = widgets.VBox([
            widgets.HBox([self.dropdown_start, self.dropdown_goal]), 
            self.dropdown_algo, 
            self.btn_run
        ])

    def run_search(self, b):
        """Callback function to execute the selected search algorithm."""
        with self.out:
            self.out.clear_output(wait=True)
            start_node = self.dropdown_start.value
            goal_node = self.dropdown_goal.value
            algo = self.dropdown_algo.value
            
            # Performance benchmarking: start timer
            t_start = time.time()

            path = []
            # Algorithm selection logic
            if algo == 'BFS (Uninformed)':
                path = bfs_search(self.G, start_node, goal_node)
            elif algo == 'DFS (Uninformed)':
                path = dfs_search(self.G, start_node, goal_node)
            elif algo == 'A* (Informed)':
                # For this, ensure pos is globally defined or passed here
                path = astar_search(self.G, start_node, goal_node, pos)

            t_end = time.time()
            execution_time = (t_end - t_start) * 1000

            if path:
                # Path formatting for display
                path_str = ' -> '.join(map(str, path))
                print(f"Path Found: {path_str}")
                print(f"Execution time: {execution_time:.4f} ms")
                
                # Calculate and display total reactance cost for any found path
                cost = sum(self.G[path[i]][path[i+1]]['weight'] for i in range(len(path)-1))
                print(f"Total Reactance: {cost:.4f} p.u.")
                
                # Trigger the visualization function
                plot_large_power_network(self.G, self.pos, path=path, title=f"Results: {algo}")
            else:
                print("[ERROR] No path found between the selected nodes.")

# Initialization and Display
search_app = LargeSearchInterface(G, pos, list(G.nodes))
display(search_app.ui, search_app.out)

## Conclusão

A implementação dos algoritmos de busca aplicada aos sistemas IEEE 14-Bus e IEEE 300-Bus apresentou resultados consistentes e satisfatórios. Inicialmente, a validação no sistema IEEE 14-Bus permitiu verificar o correto funcionamento das abordagens em um ambiente controlado. Em seguida, a aplicação no sistema IEEE 300-Bus evidenciou a robustez das soluções, mantendo desempenho adequado mesmo diante do aumento significativo da complexidade da rede.

As ferramentas de visualização desenvolvidas também se mostraram eficazes, permitindo não apenas a validação da topologia, mas também o destaque claro dos caminhos encontrados. Esse aspecto foi fundamental para a análise qualitativa dos resultados e para a verificação da coerência das rotas obtidas, especialmente em cenários com alta densidade de conexões.

A escolha dos algoritmos e da heurística adotada para o A* mostrou-se adequada ao problema proposto. A utilização da reatância como custo e da distância euclidiana como heurística permitiu equilibrar eficiência computacional e coerência física, direcionando a busca para soluções mais relevantes do ponto de vista elétrico.

De forma geral, o problema de otimização de rotas e análise de conectividade foi resolvido de maneira satisfatória, evidenciando que algoritmos clássicos de busca podem ser aplicados com sucesso a problemas reais de engenharia. Os resultados demonstram que, mesmo abordagens relativamente simples, quando bem modeladas, são capazes de fornecer suporte útil à análise e operação de sistemas elétricos.

Como perspectivas futuras, destaca-se a possibilidade de incorporar modelos mais detalhados de fluxo de potência, considerar múltiplas contingências (N-k) e explorar heurísticas mais sofisticadas. Além disso, a integração com dados em tempo real poderia ampliar ainda mais a aplicabilidade prática da abordagem proposta em ambientes operacionais.